# Lean-21 : La conjecture de Freiman-Ruzsa polynomiale (PFR) — Digestion pédagogique

**Série** : SymbolicAI / Lean — Digestions de résultats profonds
**Conjecture** : Katalin Marton (années 2010, rendue publique par Green-Tao)
**Preuve** : Terence Tao, novembre 2023
**Lac source** : https://github.com/teorth/pfr (formalisation collaborative Lean 4)
**Blog de référence** : https://terrytao.wordpress.com/2023/11/13/on-a-conjecture-of-marton/

## Navigation

| Notebook précédent | Notebook suivant |
|---|---|
| [Lean-20 - Analysis-I Tao Workflow](Lean-20-Analysis-I-Tao-Workflow.ipynb) | Lean-22 (à venir) |

---

## Présentation

Ce notebook digère la **conjecture de Freiman-Ruzsa polynomiale (PFR)** et sa preuve par Terence Tao (2023), telle que formalisée en Lean 4 dans le lac collaboratif [teorth/pfr](https://github.com/teorth/pfr). C'est le troisième volet de nos digestions de résultats profonds, après Sendov (Lean-19) et le lac Analysis-I de Tao (Lean-20) : là où Lean-19 montrait un **théorème isolé** et Lean-20 un **chantier pluriannuel**, PFR illustre un **projet collaboratif court et fini** — une preuve papier de novembre 2023, formalisée en trois semaines par une équipe distribuée, avec un blueprint public.

**La conjecture PFR** dit, en une phrase : *si A est un sous-ensemble non vide de l'espace vectoriel binaire F₂ⁿ tel que |A + A| ≤ K·|A|, alors A peut être recouvert par au plus 2K¹² classes (cosets) d'un sous-espace H de F₂ⁿ de cardinalité au plus |A|.*

**Pourquoi ce notebook dans notre série Lean ?**

- Une **méthode transmissible** : la preuve passe par la **théorie de l'entropie de Shannon** appliquée à la combinatoire additive — exactement le genre d'idée qu'un cours peut faire passer (sections 3 et 5).
- Une **mise en perspective de la formalisation** : 72 modules, des lemmes d'entropie développés pour l'occasion (`PFR/ForMathlib/`), un blueprint vivant — et un théorème final dont les seuls axiomes sont `propext`, `Classical.choice`, `Quot.sound` (section 4).
- Un **gradient de difficulté** : de la conjecture classique (borne 2K¹²) aux raffinements (exposant 11, puis 9) qui montrent une recherche vivante.

Ce notebook est le **squelette** (sections 1-4) ; le corps (sections 5-9, exercices) fera l'objet d'un second grain.

## 1. Énoncé de la conjecture

### 1.1 La version combinatoire (celle du titre)

Soit **G = F₂ⁿ** le groupe abélien des suites binaires de longueur n, avec l'addition bit à bit (modulo 2). Pour A ⊆ G non vide et K ≥ 1 un réel, on suppose que la **somme de Schurried** A + A = {a + a' | a, a' ∈ A} est **petite** : |A + A| ≤ K · |A|.

**Conjecture (Marton, PFR)** : il existe un **sous-espace vectoriel H ≤ G** tel que
- |H| ≤ |A| (H n'est pas plus gros que A), et
- A est recouvert par **au plus 2K¹² cosets** de H : A ⊆ C + H avec |C| < 2K¹².

La borne « 2K¹² » est le cœur : le nombre de cosets est **polynomial** en K — c'est le sens du mot « polynomiale » dans le nom (section 2).

### 1.2 La version entropique (celle que la preuve utilise)

La preuve de Tao ne raisonne pas sur des ensembles mais sur des **variables aléatoires**. On introduit la **distance de Ruzsa** d[X ; Y] entre deux variables aléatoires à valeurs dans G — une distance « informationnelle » qui mesure combien X et Y se ressemblent en termes d'entropie (section 3). La conjecture se reformule alors : si X₀₁ et X₀₂ sont indépendantes et presque uniformes (p.η = 1/9), il existe un sous-espace H et une variable U uniforme sur H telle que

```
d[X₀₁ ; U] + d[X₀₂ ; U] ≤ 11 · d[X₀₁ ; X₀₂]
```

C'est **cette** version que le lac `teorth/pfr` formalise en premier (théorème `entropic_PFR_conjecture`), avant d'en déduire la version combinatoire 1.1 (`PFR_conjecture`) — l'énoncé « entropique ⇒ combinatoire » est lui-même un lemme du lac (section 4).

### 1.3 Évolution de la borne

Le projet ne s'arrête pas à 2K¹² : une deuxième étape (argument de Jyun-Jie Liao) réduit l'exposant à 11, puis un raffinement récent le pousse à **9**. Le lac maintient les trois versions, ce qui en fait un excellent objet d'étude de l'évolution d'une preuve.

Voir l'énoncé en pseudo-Lean dans la cellule suivante, puis la déclaration Lean réelle en section 3.

In [1]:
# Code 1.1 - Énoncé PFR (version combinatoire) en pseudo-Lean
#
# L'énoncé réel du lac teorth/pfr (PFR/Main.lean) est :
#
#   theorem PFR_conjecture {G : Type*} [AddCommGroup G] [Module (ZMod 2) G]
#       [Fintype G] {A : Set G} {K : ℝ} (hA0 : A.Nonempty)
#       (hA : (A + A).ncard <= K * A.ncard) :
#     exists H : Submodule (ZMod 2) G,
#       exists C : Set G, C.ncard < 2 * K ^ 12
#         /\ H.ncard <= A.ncard /\ A <= C + H
#
# On reproduit la STRUCTURE (hypothèses -> conclusion) avec une fonction
# Python documentée, pour la lisibilité avant le vrai #check Lean (Code 3.1).

def pfr_statement(G, A, K):
    """
    Conjecture PFR (version combinatoire, formalisation Lean 4).

    Paramètres :
        G : type (groupe additif binaire F2^n, module sur ZMod 2)
        A : Set G — sous-ensemble non vide
        K : réel >= 1 — rapport de croissance |A+A| / |A|

    Hypothèses :
        hA0 : A.Nonempty (A non vide)
        hA  : (A + A).ncard <= K * A.ncard (somme de Schurried petite)

    Conclusion (Lean : exists H, exists C, ...) :
        H : Submodule (ZMod 2) G, |H| <= |A|
        C : Set G, |C| < 2 * K ^ 12  (au plus 2K^12 cosets)
        A ⊆ C + H                   (A recouvert par les cosets de H)
    """
    # Pseudo-Lean :
    # theorem PFR_conjecture (hA0 : A.Nonempty)
    #     (hA : (A + A).ncard <= K * A.ncard) :
    #   exists H : Submodule (ZMod 2) G, exists C : Set G,
    #     C.ncard < 2 * K ^ 12 /\ H.ncard <= A.ncard /\ A <= C + H
    #
    # Preuve : voir PFR/Main.lean du lac teorth/pfr (L253)
    # Refinements : exposant 11 (PFR_conjecture_improv), puis 9 (better_PFR_conjecture)
    pass  # stub pédagogique (règle C.1 - pas de raise)

print("PFR - version combinatoire : |A+A| <= K|A|  =>  A recouvert par < 2K^12 cosets")
print("Lac teorth/pfr : 72 modules, 3 exposants maintenus (12, 11, 9), preuve sans sorry")
print("Theoreme phare : PFR.Main.PFR_conjecture (voir Code 3.1 pour le #check Lean reel)")

PFR - version combinatoire : |A+A| <= K|A|  =>  A recouvert par < 2K^12 cosets
Lac teorth/pfr : 72 modules, 3 exposants maintenus (12, 11, 9), preuve sans sorry
Theoreme phare : PFR.Main.PFR_conjecture (voir Code 3.1 pour le #check Lean reel)


## 2. « Polynomiale » — ce qui change par rapport à Freiman-Ruzsa classique

### 2.1 La conjecture de Freiman-Ruzsa (théorie additive classique)

Dans ℤ (ou un groupe abélien général), le **théorème de Freiman** affirme : si A est fini et |A + A| ≤ K·|A|, alors A est contenu dans l'union d'un petit nombre de **progressions arithmétiques** de taille bornée (fonction de K seule, pas de |A|). La **conjecture de Freiman-Ruzsa** demande un contrôle *quantitatif* : combien de progressions, de quelle taille, en fonction de K ?

La forme la plus connue, pour ℤ, donne des bornes de l'ordre de **2^{K⁴}** (Ruzsa) : le nombre de progressions est **exponentiel** en K. Remplacer cette borne par une borne **polynomiale** est resté ouvert des décennies — d'où l'importance de la conjecture.

### 2.2 La version PFR (dans F₂ⁿ)

Dans l'espace **F₂ⁿ**, les « progressions arithmétiques » du cas ℤ deviennent des **sous-espaces vectoriels** (la somme A+A ≅ A est le symétrique de la structure linéaire). La conjecture de Marton demande alors :

> si |A + A| ≤ K·|A|, alors A tient dans **polynômie en K** cosets d'un sous-espace pas plus gros que A.

C'est le passage d'exponentiel (2^{K⁴}) à polynomial (2K¹²) qui mérite le nom de **polynomiale** : le coût de la recouverte ne gonfle plus de manière exponentielle quand K augmente.

### 2.3 Pourquoi F₂ⁿ est-il le bon terrain ?

- Le groupe F₂ⁿ est **2-torsion** : 2x = 0 pour tout x — la structure est « plate », sans progression arithmétique longue, ce qui force à raisonner par sous-espaces plutôt que par segments.
- La **borne 2K¹²** est indépendante de n : la dimension n ne joue aucun rôle dans la constante. C'est une propriété remarquable (et fragile) de la conjecture.
- La démonstration utilise l'**entropie** (section 3), qui se comporte particulièrement bien sur les variables à valeurs dans un groupe de torsion.

La cellule suivante illustre le plongement : un petit A dans F₂³, sa somme A+A, et le coset-revêtement par un sous-espace.

In [2]:
# Code 2.1 - Illustration : A + A et coset-revêtement dans F2^3
#
# Exemple concret : A = {(0,0,0), (1,0,0), (0,1,0)} dans F2^3.
#   |A| = 3 ; A + A = {0, (1,0,0), (0,1,0), (1,1,0)} -> |A+A| = 4 <= 2*3 (K=2).
#   Sous-espace H = Vect{(1,0,0)} (cardinalite 2 <= |A|) ; cosets H et (0,1,0)+H.

def f2(n):  # liste des elements de F2^n (entiers -> vecteurs binaires)
    return [tuple((i >> k) & 1 for k in range(n)) for i in range(2 ** n)]

def add(u, v):  # addition bit a bit modulo 2
    return tuple((a + b) % 2 for a, b in zip(u, v))

def schurried_sum(A):
    return sorted({add(a, b) for a in A for b in A})

A = {(0, 0, 0), (1, 0, 0), (0, 1, 0)}
S = schurried_sum(A)
H = {(0, 0, 0), (1, 0, 0)}                 # sous-espace Vect{(1,0,0)}
cosets = [sorted({add(h, c) for h in H}) for c in [(0, 0, 0), (0, 1, 0)]]
K = 2.0

print(f"|A| = {len(A)}, A+A = {S}, |A+A| = {len(S)}")
print(f"K = |A+A|/|A| = {len(S)}/{len(A)} = {len(S)/len(A):.2f}  (hypothese <= {K})")
print(f"H = {sorted(H)} (|H| = {len(H)} <= |A| = {len(A)})")
print(f"cosets de H : {cosets}")
print(f"reunion = {sorted(cosets[0] + cosets[1])} ; A inclus ? {set(cosets[0] + cosets[1]) >= A}")
print(f"nb de cosets = 2 < 2*K^12 = {2 * K ** 12:.0f} : borne polynomiale verifiee")
print("NB : borne polynomiale en K independante de n (dimension 3 ici, le principe tient)")

|A| = 3, A+A = [(0, 0, 0), (0, 1, 0), (1, 0, 0), (1, 1, 0)], |A+A| = 4
K = |A+A|/|A| = 4/3 = 1.33  (hypothese <= 2.0)
H = [(0, 0, 0), (1, 0, 0)] (|H| = 2 <= |A| = 3)
cosets de H : [[(0, 0, 0), (1, 0, 0)], [(0, 1, 0), (1, 1, 0)]]
reunion = [(0, 0, 0), (0, 1, 0), (1, 0, 0), (1, 1, 0)] ; A inclus ? True
nb de cosets = 2 < 2*K^12 = 8192 : borne polynomiale verifiee
NB : borne polynomiale en K independante de n (dimension 3 ici, le principe tient)


## 3. L'entropie dans un énoncé purement combinatoire

### 3.1 Le pli informationnel

L'énoncé de la section 1 ne parle que d'ensembles et de cardinaux. Pourquoi l'**entropie de Shannon** intervient-elle ? L'idée de base : si X est une variable aléatoire uniforme sur A, alors H(X) = log₂|A| — l'entropie **est** le cardinal, vu à travers le logarithme. La croissance |A + A| ≤ K·|A| se traduit alors en termes d'entropie de sommes de variables indépendantes :

- H(X + X') ≤ log K + H(X), où X' est une copie indépendante de X ;
- l'**inégalité de Ruzsa** : H(X + X') ≤ 2H(X) − H(X − X') + O(1), qui relie somme et différence.

### 3.2 La distance de Ruzsa

Pour mesurer « combien X et Y se ressemblent », on définit la **distance de Ruzsa**

```
d[X ; Y] = H(X') + H(Y') − H(X' + Y')
```

où X', Y' sont des copies indépendantes. C'est une distance (symétrique, vérifie l'inégalité triangulaire) qui ne dépend que des lois, pas des supports. Un théorème central de la preuve est le **théorème de réduction de la distance** (Ruzsa distance inequality) : si d[X ; X'] est très petite, alors X est presque uniforme sur un sous-groupe. C'est le mécanisme par lequel la preuve passe de « deux variables presque égales » à « un sous-espace uniforme ».

### 3.3 La conjecture entropique (déclaration Lean)

La version entropique de PFR — celle que le lac prouve en premier — s'énonce avec la distance de Ruzsa :

```
theorem entropic_PFR_conjecture (hpη : p.η = 1/9) :
    ∃ H : Submodule (ZMod 2) G, ∃ Ω, ∃ U : Ω → G,
      IsProbabilityMeasure ℙ ∧ Measurable U ∧ IsUniform H U ∧
      d[p.X₀₁ # U] + d[p.X₀₂ # U] ≤ 11 * d[p.X₀₁ # p.X₀₂]
```

Le paramètre `p.η = 1/9` fixe un régime de « presque uniformité » ; `d[X # Y]` est la distance de Ruzsa (notation Lean du lac). La cellule suivante **vérifie la déclaration réelle** dans le lac compilé : c'est la sortie du vrai Lean, pas une reproduction.

In [3]:
# Code 3.1 - #check REEL : la declaration entropique de PFR dans le lac compile
#
# On execute `lake env lean` sur un snippet qui importe le lac teorth/pfr
# et interroge les declarations reelles. Le chargement de l'environnement
# (imports Mathlib) prend quelques minutes la premiere fois.
# Le lac doit etre present : setup ci-dessous le clone s'il manque.

import json
import os, subprocess, sys
from pathlib import Path

PROJECT_NAME = "pfr_lean"
# Chemin du lac : variable d'env explicite, sinon dossier a cote du notebook.
CANDIDATES = [
    Path(os.environ.get("PFR_LEAN_PATH", "")),
    Path.cwd() / PROJECT_NAME,
    Path.cwd().parent / PROJECT_NAME,
]
LAKE_DIR = next((p for p in CANDIDATES if p and p.exists() and (p / "lakefile.toml").exists()), None)

if LAKE_DIR is None:
    # Repli : telecharger le lac (clone leger) puis preparer le cache Mathlib.
    # NB : `lake exe cache get` telecharge ~1-2 Go d'oleans la premiere fois.
    print("[setup] Lac PFR absent : clone depuis teorth/pfr (depth 1)...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/teorth/pfr.git", str(Path.cwd() / PROJECT_NAME)],
                   check=True)
    LAKE_DIR = Path.cwd() / PROJECT_NAME
    print("[setup] Preparation du cache Mathlib (long la premiere fois)...")
    subprocess.run(["lake", "exe", "cache", "get"], cwd=LAKE_DIR, check=True)
    print("[setup] Build des oleans PFR (les #check du notebook en ont besoin)...")
    subprocess.run(["lake", "build"], cwd=LAKE_DIR, check=True)
else:
    print(f"[setup] Lac PFR trouve : {LAKE_DIR}")

SNIPPET = """import PFR
import PFR.EntropyPFR
import PFR.Main

-- La conjecture entropique (noyau de la preuve, section 3)
#check entropic_PFR_conjecture
-- La version combinatoire finale (exposant 12, section 1)
#check PFR_conjecture
-- Les axiomes de la preuve (proprete formelle, section 4)
#print axioms PFR_conjecture
"""

tmp = Path(os.environ.get("TMP", "/tmp")) / "lean21_pfr_check.lean"
tmp.write_text(SNIPPET, encoding="utf-8")
print("--- lake env lean (chargement de l'environnement PFR, 1-3 min) ---")
res = subprocess.run(["lake", "env", "lean", str(tmp)],
                     cwd=str(LAKE_DIR), capture_output=True, text=True, timeout=600)
print(res.stdout)
print(res.stderr)
tmp.unlink(missing_ok=True)
print("Sortie ci-dessus : declarations RELLES du lac teorth/pfr (compilation locale).")

[setup] Lac PFR trouve : C:\dev\_lakes\pfr
--- lake env lean (chargement de l'environnement PFR, 1-3 min) ---


entropic_PFR_conjecture.{uG, u_1, u_2} {Ω₀₁ : Type u_1} {Ω₀₂ : Type u_2} [MeasureTheory.MeasureSpace Ω₀₁]
  [MeasureTheory.MeasureSpace Ω₀₂] [MeasureTheory.IsProbabilityMeasure MeasureTheory.volume]
  [MeasureTheory.IsProbabilityMeasure MeasureTheory.volume] {G : Type uG} [AddCommGroup G] [Module (ZMod 2) G]
  [Finite G] [MeasurableSpace G] [MeasurableSingletonClass G] (p : refPackage Ω₀₁ Ω₀₂ G) (hpη : p.η = 1 / 9) :
  ∃ H Ω mΩ U,
    MeasureTheory.IsProbabilityMeasure MeasureTheory.volume ∧
      Measurable U ∧
        ProbabilityTheory.IsUniform (↑H) U MeasureTheory.volume ∧ d[p.X₀₁ # U] + d[p.X₀₂ # U] ≤ 11 * d[p.X₀₁ # p.X₀₂]
PFR_conjecture.{u_1} {G : Type u_1} [AddCommGroup G] {A : Set G} {K : ℝ} [Countable G] [Module (ZMod 2) G] [Finite G]
  (hA₀ : A.Nonempty) (hA : ↑(A + A).ncard ≤ K * ↑A.ncard) :
  ∃ H c, ↑(Nat.card ↑c) < 2 * K ^ 12 ∧ (↑H).ncard ≤ A.ncard ∧ A ⊆ c + ↑H
'PFR_conjecture' depends on axioms: [propext, Classical.choice, Quot.sound]


Sortie ci-dessus : declarations REL

## 4. Architecture du lac `teorth/pfr`

### 4.1 Un projet collaboratif court et fini

Lancé mi-novembre 2023 pour formaliser la preuve de Tao, le lac `pfr` a atteint son but en **trois semaines** — un cas d'école de formalisation rapide : un blueprint public, un canal Zulip dédié, une équipe distribuée, et des lemmes « for Mathlib » conçus pour être réintégrés. En 2026, le dépôt continue d'évoluer (extension aux groupes de torsion bornés, raffinement de l'exposant).

### 4.2 Les modules

72 fichiers Lean, organisés ainsi :

- **`PFR/EntropyPFR.lean`** — la conjecture entropique et sa preuve par la fonctionnelle τ (le « cœur »).
- **`PFR/Main.lean`** — la version combinatoire finale (`PFR_conjecture`, exposant 12) et sa preuve « entropique ⇒ combinatoire ».
- **`PFR/ImprovedPFR.lean`** — les raffinements (exposant 11, puis 9).
- **`PFR/ForMathlib/`** — ~20 modules d'entropie généralisés (distance de Ruzsa, information mutuelle, indépendance conditionnelle) rédigés dans le style Mathlib pour être contribués en amont.
- **`PFR/FirstEstimate.lean`, `Fibring.lean`, `Endgame.lean`** — les étapes techniques de la preuve.

### 4.3 Les dépendances Mathlib

Le lac utilise directement le noyau de probabilité de Mathlib : `Mathlib.Probability` (mesures de probabilité, variables aléatoires), et développe `ProbabilityTheory.entropy` et `ProbabilityTheory.rdist` (distance de Ruzsa) dans `PFR/ForMathlib/`. Deux dépendances tierces : `AddCombi` (combinatoire additive) et `checkdecls` (outil de vérification des déclarations par le blueprint).

### 4.4 Propreté formelle

Le théorème final est prouvé **sans `sorry`** : `#print axioms PFR_conjecture` n'affiche que les trois axiomes de base du calcul des constructions (propext, choix classique, Quot.sound). La cellule suivante vérifie cette propriété sur la compilation locale — c'est la garantie « c'est une vraie preuve », pas une esquisse.

In [4]:
# Code 4.1 - Propreté formelle : #print axioms sur la compilation locale
#
# Meme mecanique que le Code 3.1, mais on verifie SPECIFIQUEMENT que la
# preuve de PFR n'utilise ni `sorry` ni axiome d'amelioration (regle
# anti-regression D du depot : preuve formelle = preuve complete).
# Le champ `PFR.Examples` du lac contient la version self-contained ;
# on interroge ici la declaration du module Main.

import os, subprocess
from pathlib import Path

# Reutilise le LAKE_DIR detecte au Code 3.1 (cellule precedente).
LAKE_DIR = next((p for p in CANDIDATES if p and p.exists() and (p / "lakefile.toml").exists()), None)
assert LAKE_DIR is not None, "Executer d'abord le Code 3.1 (detection du lac)"

SNIPPET = """import PFR.Main
import PFR.ImprovedPFR
import PFR.RhoFunctional

-- Exposant 12 : axiomes de la preuve finale (attendu : propext, choice, Quot)
#print axioms PFR_conjecture
-- Exposant 11 (Liao) : meme propreté
#print axioms PFR_conjecture_improv
-- Exposant 9 (refinement actuel)
#print axioms better_PFR_conjecture
"""

tmp = Path(os.environ.get("TMP", "/tmp")) / "lean21_pfr_axioms.lean"
tmp.write_text(SNIPPET, encoding="utf-8")
print("--- #print axioms sur les trois exposants (lac compile) ---")
res = subprocess.run(["lake", "env", "lean", str(tmp)],
                     cwd=str(LAKE_DIR), capture_output=True, text=True, timeout=600)
print(res.stdout)
print(res.stderr)
tmp.unlink(missing_ok=True)

print()
print("Squelette Lean-21 termine (sections 1-4). Corps (sections 5-9 + exercices) : grain 3.2.")
print("Voir issue #10932 pour le plan complet et #10763 (Epic digestions Tao).")

--- #print axioms sur les trois exposants (lac compile) ---


'PFR_conjecture' depends on axioms: [propext, Classical.choice, Quot.sound]
'PFR_conjecture_improv' depends on axioms: [propext, Classical.choice, Quot.sound]
'better_PFR_conjecture' depends on axioms: [propext, Classical.choice, Quot.sound]



Squelette Lean-21 termine (sections 1-4). Corps (sections 5-9 + exercices) : grain 3.2.
Voir issue #10932 pour le plan complet et #10763 (Epic digestions Tao).
